In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, AsyncOpenAI, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from job_hunter import JobHunter
from langsmith.wrappers import OpenAIAgentsTracingProcessor
from agents import set_trace_processors
import os
from templates import job_search_agent_instruction

set_trace_processors([OpenAIAgentsTracingProcessor()])

load_dotenv(override=True)

True

In [ ]:
client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1", 
    api_key=os.getenv("GROQ_API_KEY")
)

model = OpenAIChatCompletionsModel(model="google/gemini-2.5-flash", openai_client=client)

In [ ]:
job_hunter = JobHunter()
await job_hunter.search_jobs("Remote AI Engineer")

In [ ]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

In [ ]:
job_hunter_params = {"command": "uv", "args": ["run", "job_hunter_server.py"]}
async with MCPServerStdio(params=job_hunter_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

In [ ]:
instructions = job_search_agent_instruction()

async def run_job_hunter(request: str):
    async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
        async with MCPServerStdio(params=job_hunter_params, client_session_timeout_seconds=60) as mcp_server_jobs:
            agent = Agent(
                name="job_search_expert", 
                instructions=instructions, 
                model=model,
                mcp_servers=[mcp_server_jobs, mcp_server_files]
            )
            
            print("🚀 Starting job hunt...")
            with trace("job_hunting_workflow"):
                result = await Runner.run(agent, request, max_turns=30)
            display(Markdown(result.final_output))


In [ ]:
request = """
Find me 10 'Senior AI Engineer' jobs that are remote-friendly. I need you to successfully process and create at least 5 separate .md files for the best matches.

My Profile:
- Name: Sodiq Alabi
- Skills: Python, TensorFlow, LLM Orchestration, Scalable ML solutions.

If you hit dead links or login walls, please keep searching until you hit the minimum target of 5 files.
"""

await run_job_hunter(request)